# Unit 5: Evaluation of Spatial Predictions

This notebook supports the Unit 5 ILOs:

- compute accuracy metrics for spatial predictions,
- interpret model performance in a geospatial context,
- compare the effectiveness of different AI approaches.

It is designed to evaluate outputs from the wildfire, flood, and landslide units, but it can also be used with placeholder prediction rasters.

## How To Interpret This Notebook

This notebook is not only a collection of formulas. It is a guide to thinking critically about spatial prediction quality. In hazard mapping, a model can look strong according to one metric and still perform poorly in a way that matters for the real application. For example, overall accuracy can be high even when the hazard pixels are detected poorly, simply because the background dominates the image.

Students should therefore treat each metric as one lens on model behavior. Precision, recall, F1, IoU, and balanced accuracy each emphasize something different. The most important skill is learning how to connect those numbers back to the map and to the hazard-management purpose.

## 1. Why evaluation is different in geospatial AI

Pixel-wise accuracy alone can be misleading. Hazard maps are often spatially imbalanced, geographically clustered, and sensitive to edge uncertainty. A useful evaluation therefore combines classification metrics with geospatial interpretation.

In [ ]:
import numpy as np
import pandas as pd

try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    roc_auc_score = None

print('Metric notebook initialized')

In [ ]:
rng = np.random.default_rng(99)
y_true = rng.integers(0, 2, size=5000)
y_prob = np.clip(0.15 + 0.7 * y_true + rng.normal(0, 0.2, size=5000), 0, 1)
y_pred = (y_prob >= 0.5).astype(int)
y_true[:10], y_prob[:10], y_pred[:10]

In [ ]:
def safe_divide(a, b):
    return a / b if b != 0 else np.nan

def confusion_counts(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tp, tn, fp, fn

def compute_metrics(y_true, y_pred, y_prob=None):
    tp, tn, fp, fn = confusion_counts(y_true, y_pred)
    metrics = {
        'accuracy': safe_divide(tp + tn, tp + tn + fp + fn),
        'precision': safe_divide(tp, tp + fp),
        'recall': safe_divide(tp, tp + fn),
        'f1': safe_divide(2 * tp, 2 * tp + fp + fn),
        'iou': safe_divide(tp, tp + fp + fn),
        'specificity': safe_divide(tn, tn + fp),
        'balanced_accuracy': np.nanmean([safe_divide(tp, tp + fn), safe_divide(tn, tn + fp)]),
    }
    if y_prob is not None and roc_auc_score is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
    return pd.Series(metrics)

compute_metrics(y_true, y_pred, y_prob)

## 2. Spatial interpretation guide

The same metric value can mean different things depending on the hazard and use case.

- High recall and lower precision may be acceptable in emergency screening.
- High IoU is important when map geometry matters.
- Balanced accuracy matters when hazard pixels are rare.
- Spatially clustered errors can still be problematic even when global metrics look good.

In [ ]:
def compare_models(results_dict):
    rows = []
    for model_name, payload in results_dict.items():
        row = compute_metrics(payload['y_true'], payload['y_pred'], payload.get('y_prob'))
        row.name = model_name
        rows.append(row)
    return pd.DataFrame(rows).sort_values('iou', ascending=False)

example_results = {
    'RandomForest': {'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob},
    'CNN': {'y_true': y_true, 'y_pred': np.where(y_prob > 0.45, 1, 0), 'y_prob': y_prob},
    'Transformer': {'y_true': y_true, 'y_pred': np.where(y_prob > 0.55, 1, 0), 'y_prob': y_prob},
}

compare_models(example_results)

## Suggested extension for students

1. Import predictions from previous units and compare metrics.
2. Check whether the best global model also gives the best map boundaries.
3. Compare performance under different probability thresholds.
4. Discuss which metric is most appropriate for the hazard management objective.

## Unit 5 Takeaway

The most important lesson from this notebook is that evaluation is interpretive as well as computational. A good hazard map should be judged not only by one score, but by how different metrics describe its strengths and weaknesses and by how those errors appear spatially on the map.

Students should leave this unit understanding that model comparison becomes meaningful only when the metrics are tied back to the hazard context, the class balance, and the intended use of the prediction.